python mesa ile 100 ajandan oluşan bir model oluşturup ajanları AgentSet() içine alan Kod


Bu kodda:

MyAgent adında basit bir ajan sınıfı oluşturduk.

MyModel içinde 100 ajan oluşturduk ve bunları rastgele hücrelere yerleştirdik.

schedule.agents ile tüm ajanlara erişebilirsiniz (bu bir AgentSet'tir).

Modeli 10 adım çalıştırdık.

Gruplama, Sıralama yaptık


In [1]:
!pip install mesa[rec]
!pip install seaborn

# Has multi-dimensional arrays and matrices.
# Has a large collection of mathematical functions to operate on these arrays.
import numpy as np

# Data manipulation and analysis.
import pandas as pd

# Data visualization tools.
import seaborn as sns

import mesa



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.8/265.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.9 MB/s eta 0:00:00


In [182]:

from mesa.space import MultiGrid
from mesa.datacollection import DataCollector
import random

# Ajan sınıfını tanımlayalım
class MyAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.active = random.choice([True, False])
        self.wealth = random.randint(0, 100)
        self.group = "A" if self.unique_id % 2 == 0 else "B"  # Çift-ID'liler "A" grubu



    def say_hi(self):
        print(
            f"Merhaba! Ben {self.unique_id}, "
            f"Servetim: {self.wealth}, "
            f"Aktif mi? {self.active}, "
            f"Grubum: {self.group}"
        )


    def step(self):
        # Ajanın her adımda yapacağı işlemler
        print(f"Ajan ID: {self.unique_id}- Servet: {self.wealth}")
        self.say_hi()

# Model sınıfını tanımlayalım
class MyModel(mesa.Model):
    def __init__(self, N, width, height):
        super().__init__()
        self.num_agents = N
        self.grid = MultiGrid(width, height, True)


        # Ajanları oluşturalım
        for i in range(self.num_agents):
            agent = MyAgent(self)
            self.agents.add(agent)

            # Ajanları rastgele bir hücreye yerleştirelim
            x = self.random.randrange(self.grid.width)
            y = self.random.randrange(self.grid.height)
            self.grid.place_agent(agent, (x, y))

        # Veri toplamak için DataCollector kullanabiliriz (opsiyonel)
        self.datacollector = DataCollector()



    # Modelde filtreleme:
    def get_group_a(self):
        return self.agents.select(lambda agent: agent.group == "A")


    # Dinamik Ajan Ekleyip Çıkarma

    def dynamic_agents(self):
        # diğer piyasalara ilişkin kötü haberler arttığında ve mevcut piyasaya ilşkin iyi haberler arttığında, piyasaya ajan girişi diğer zamanlardaki
        # girişlerden daha çok olacaktır. Bu nedenle piyasaya gelen haberlere göre ajan girişi olacak
        if random.random() > 0.5: #iyi haber geldi, 100 yeni ajan girdi
            # Yeni ajan ekle
            for i in range(int(0.25 * len(self.agents))):   # yeni giriş yapan ajanların sayısını, mevcut ajan sayısının belli bir oranı olarak ayarlanabilir.
                new_agent = MyAgent(self)
                self.agents.add(new_agent)
            print(f"Yeni eklenen ajan SAYISI: {(int(0.25 * len(self.agents)))}")
        else:                    # iyi haber yok, 10 yeni ajan girdi
            # Yeni ajan ekle
            for i in range(int(0.10 * len(self.agents))):
                new_agent = MyAgent(self)
                self.agents.add(new_agent)
            print(f"Yeni eklenen ajan sayısİ: {(int(0.10 * len(self.agents)))}")

        print(f"Güncel Ajan SAyısı: {len(self.agents)}")

        """
        # 5 ID'li ajanı sil
        agent_to_remove = next(a for a in self.agents if a.unique_id == 5)
        self.agents.remove(agent_to_remove)
        """




    def step(self):
        # Modelin her adımda yapacağı işlemler
        self.agents.do("step")
        self.dynamic_agents()

        self.datacollector.collect(self)

# Modeli oluşturalım (100 ajan, 10x10 grid)
model = MyModel(10, 10, 10)

# Tüm ajanları AgentSet olarak almak için:
all_agents = model.agents

# AgentSet'i kontrol edelim
print(f"Toplam ajan sayısı: {len(all_agents)}")
print(f"İlk ajanın ID'si: {all_agents[0].unique_id}")

# Modeli birkaç adım çalıştıralım
for i in range(10):
    print(f"------Adım {i+1}--------")
    model.step()

Toplam ajan sayısı: 10
İlk ajanın ID'si: 1
------Adım 1--------
Ajan ID: 1- Servet: 4
Merhaba! Ben 1, Servetim: 4, Aktif mi? False, Grubum: B
Ajan ID: 2- Servet: 97
Merhaba! Ben 2, Servetim: 97, Aktif mi? True, Grubum: A
Ajan ID: 3- Servet: 7
Merhaba! Ben 3, Servetim: 7, Aktif mi? True, Grubum: B
Ajan ID: 4- Servet: 100
Merhaba! Ben 4, Servetim: 100, Aktif mi? True, Grubum: A
Ajan ID: 5- Servet: 50
Merhaba! Ben 5, Servetim: 50, Aktif mi? True, Grubum: B
Ajan ID: 6- Servet: 11
Merhaba! Ben 6, Servetim: 11, Aktif mi? False, Grubum: A
Ajan ID: 7- Servet: 60
Merhaba! Ben 7, Servetim: 60, Aktif mi? False, Grubum: B
Ajan ID: 8- Servet: 50
Merhaba! Ben 8, Servetim: 50, Aktif mi? False, Grubum: A
Ajan ID: 9- Servet: 10
Merhaba! Ben 9, Servetim: 10, Aktif mi? True, Grubum: B
Ajan ID: 10- Servet: 71
Merhaba! Ben 10, Servetim: 71, Aktif mi? True, Grubum: A
Yeni eklenen ajan SAYISI: 3
Güncel Ajan SAyısı: 12
------Adım 2--------
Ajan ID: 1- Servet: 4
Merhaba! Ben 1, Servetim: 4, Aktif mi? False, Gr

In [37]:

# Mevcut ajanlardan bir AgentSet oluşturalım
#agent_set = AgentSet(model.agents)

# AgentSet üzerinde işlemler yapabiliriz
print(f"AgentSet boyutu: {all_agents[0]}")

AgentSet boyutu: <__main__.MyAgent object at 0x7eef070f2010>


In [14]:
# Access agent attributes

for agent in all_agents:
       print(agent.unique_id)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100


In [36]:
# Assuming agents have an 'active' attribute

def is_active(agent):
       return agent.active

# Use a list comprehension to filter agents
active_agents = [agent for agent in all_agents if is_active(agent)]

# active_agents now contains a list of active agents

for agent in active_agents:
       print(agent.unique_id)



1
2
4
5
7
8
9
10
14
18
20
22
24
26
28
31
33
35
39
40
41
42
44
46
49
51
52
54
55
57
61
62
69
71
74
78
80
81
90
91
93
95
99


In [54]:
# Sort by agent's 'wealth' attribute
all_agents.sort(key=lambda agent: agent.wealth, ascending=False)  # Sort using a lambda function

# Accessing the first agent after sorting (or any specific agent by index)
print(f"First Agent - Unique ID: {all_agents[0].unique_id}, Active: {all_agents[0].active}, Wealth: {all_agents[0].wealth}")


First Agent - Unique ID: 1, Active: False, Wealth: 92


In [59]:
# Sort by agent's 'wealth' attribute
all_agents.sort(key=lambda agent: agent.wealth, ascending=False)  # Sort using a lambda function

# Iterate through the sorted agents and print their attributes
for agent in all_agents:
    print(f"Agent - Unique ID: {agent.unique_id}, Active: {agent.active}, Wealth: {agent.wealth}")

Agent - Unique ID: 1, Active: False, Wealth: 92
Agent - Unique ID: 2, Active: False, Wealth: 40
Agent - Unique ID: 3, Active: False, Wealth: 9
Agent - Unique ID: 4, Active: True, Wealth: 91
Agent - Unique ID: 5, Active: False, Wealth: 3
Agent - Unique ID: 6, Active: True, Wealth: 13
Agent - Unique ID: 7, Active: False, Wealth: 91
Agent - Unique ID: 8, Active: True, Wealth: 85
Agent - Unique ID: 9, Active: True, Wealth: 84
Agent - Unique ID: 10, Active: True, Wealth: 60
Agent - Unique ID: 11, Active: True, Wealth: 36
Agent - Unique ID: 12, Active: False, Wealth: 61
Agent - Unique ID: 13, Active: True, Wealth: 73
Agent - Unique ID: 14, Active: True, Wealth: 7
Agent - Unique ID: 15, Active: True, Wealth: 11
Agent - Unique ID: 16, Active: True, Wealth: 74
Agent - Unique ID: 17, Active: True, Wealth: 90
Agent - Unique ID: 18, Active: True, Wealth: 77
Agent - Unique ID: 19, Active: False, Wealth: 37
Agent - Unique ID: 20, Active: True, Wealth: 1
Agent - Unique ID: 21, Active: True, Wealth: 8

In [53]:
random_agent = all_agents.random.choice(all_agents)
print(f"Random Agent - Unique ID: {random_agent.unique_id}, Active: {random_agent.active}, Wealth: {random_agent.wealth}")

Random Agent - Unique ID: 87, Active: False, Wealth: 24


In [60]:
# Print the unique_id of the first agent before sorting
print(f"First agent before sorting: {all_agents[0].unique_id}")

# Sort by agent's 'wealth' attribute
all_agents.sort(key=lambda agent: agent.wealth, ascending=False)

# Print the unique_id of the first agent after sorting
print(f"First agent after sorting: {all_agents[0].unique_id}")

First agent before sorting: 1
First agent after sorting: 1


In [64]:
# Print the unique_id of the first agent before sorting
print(f"First agent before sorting: {all_agents[0].unique_id}")

# Sort by agent's 'wealth' attribute
sorted_agent_with_wealth= all_agents.sort(key=lambda agent: agent.wealth, ascending=False)

# Print the unique_id of the first agent after sorting
print(f"First agent after sorting: {sorted_agent_with_wealth[0].unique_id}")

First agent before sorting: 1
First agent after sorting: 58


In [63]:
# Create a sorted copy using the sorted() function
sorted_agents = sorted(all_agents, key=lambda agent: agent.wealth, reverse=True)

# Now, sorted_agents contains the sorted agents, and all_agents remains unchanged

for agent in sorted_agents:
    print(f"Agent - Unique ID: {agent.unique_id}, Active: {agent.active}, Wealth: {agent.wealth}")

Agent - Unique ID: 58, Active: False, Wealth: 100
Agent - Unique ID: 65, Active: False, Wealth: 100
Agent - Unique ID: 23, Active: True, Wealth: 98
Agent - Unique ID: 79, Active: True, Wealth: 96
Agent - Unique ID: 1, Active: False, Wealth: 92
Agent - Unique ID: 60, Active: False, Wealth: 92
Agent - Unique ID: 71, Active: False, Wealth: 92
Agent - Unique ID: 4, Active: True, Wealth: 91
Agent - Unique ID: 7, Active: False, Wealth: 91
Agent - Unique ID: 27, Active: False, Wealth: 91
Agent - Unique ID: 17, Active: True, Wealth: 90
Agent - Unique ID: 93, Active: True, Wealth: 89
Agent - Unique ID: 34, Active: True, Wealth: 87
Agent - Unique ID: 8, Active: True, Wealth: 85
Agent - Unique ID: 25, Active: False, Wealth: 85
Agent - Unique ID: 9, Active: True, Wealth: 84
Agent - Unique ID: 67, Active: False, Wealth: 84
Agent - Unique ID: 48, Active: False, Wealth: 83
Agent - Unique ID: 49, Active: False, Wealth: 83
Agent - Unique ID: 62, Active: False, Wealth: 83
Agent - Unique ID: 59, Active: 

In [68]:
# Grubu yazdırma:
group_a = model.get_group_a()
print(f"A grubundaki ajan sayısı: {len(group_a)}")

A grubundaki ajan sayısı: 50


In [113]:
# Dinamik Ajan Ekleyip Çıkarma

model.dynamic_agents()


Güncel sayı: 104


In [118]:
# model.agents.select - koşula uyan ajanları seçer

serveti_10dan_buyuk= model.agents.select(lambda a: a.wealth > 10)
print(f"Servet 10'dan büyük olan ajan sayısı: {len(serveti_10dan_buyuk)}")

Servet 10'dan büyük olan ajan sayısı: 94


In [127]:
# ajanları alt gruplara ayır

# Calculate the number of agents for each group
num_agents_group1 = int(len(model.agents) * 0.3)
num_agents_group2 = len(model.agents) - num_agents_group1

# Create the groups using slicing
group1 = model.agents[:num_agents_group1]
group2 = model.agents[num_agents_group1:]

print("Group1:")
for agent in group1:
    print(f"  Agent - Unique ID: {agent.unique_id}, Active: {agent.active}, Wealth: {agent.wealth}")

print("\nGroup2:")
for agent in group2:
    print(f"  Agent - Unique ID: {agent.unique_id}, Active: {agent.active}, Wealth: {agent.wealth}")


# Now, group1 and group2 contain the desired subgroups of agents

Group1:
  Agent - Unique ID: 1, Active: False, Wealth: 88
  Agent - Unique ID: 2, Active: False, Wealth: 71
  Agent - Unique ID: 3, Active: True, Wealth: 72
  Agent - Unique ID: 4, Active: False, Wealth: 45
  Agent - Unique ID: 5, Active: False, Wealth: 62
  Agent - Unique ID: 6, Active: True, Wealth: 76
  Agent - Unique ID: 7, Active: True, Wealth: 31
  Agent - Unique ID: 8, Active: False, Wealth: 33
  Agent - Unique ID: 9, Active: False, Wealth: 93
  Agent - Unique ID: 10, Active: True, Wealth: 58
  Agent - Unique ID: 11, Active: False, Wealth: 50
  Agent - Unique ID: 12, Active: True, Wealth: 70
  Agent - Unique ID: 13, Active: False, Wealth: 82
  Agent - Unique ID: 14, Active: False, Wealth: 48
  Agent - Unique ID: 15, Active: True, Wealth: 45
  Agent - Unique ID: 16, Active: True, Wealth: 52
  Agent - Unique ID: 17, Active: False, Wealth: 39
  Agent - Unique ID: 18, Active: True, Wealth: 3
  Agent - Unique ID: 19, Active: True, Wealth: 4
  Agent - Unique ID: 20, Active: False, Wea

In [135]:
# İlk 5 ajanın özelliklerini yazdır
for agent in list(model.agents)[:5]:
    print(
        f"Ajan {agent.unique_id}: "
        f"Servet={agent.wealth}, "
        f"Aktif mi?={agent.active}, "

    )

Ajan 1: Servet=108, Aktif mi?=False, 
Ajan 2: Servet=70, Aktif mi?=False, 
Ajan 3: Servet=102, Aktif mi?=True, 
Ajan 4: Servet=51, Aktif mi?=True, 
Ajan 5: Servet=89, Aktif mi?=False, 


In [136]:
# Tüm ajanların özelliklerine erişip değer atama/ değişiklik yapma
# Tüm ajanların servetini artırma
for agent in model.agents:
    agent.wealth += 10

for agent in list(model.agents)[:5]:
    print(
        f"Ajan {agent.unique_id}: "
        f"Servet={agent.wealth}, "
        f"Aktif mi?={agent.active}, "

    )

Ajan 1: Servet=118, Aktif mi?=False, 
Ajan 2: Servet=80, Aktif mi?=False, 
Ajan 3: Servet=112, Aktif mi?=True, 
Ajan 4: Servet=61, Aktif mi?=True, 
Ajan 5: Servet=99, Aktif mi?=False, 


In [159]:
# Ajanları belli bir kritere göre gruplamak

rich_agents = []

for agent in model.agents:
    if agent.wealth > 80:
        rich_agents.append(agent)
        print(
            f"Ajan {agent.unique_id}: "
            f"Servet={agent.wealth}, "
            f"Aktif mi?={agent.active}, "
            )


Ajan 1: Servet=118, Aktif mi?=False, 
Ajan 3: Servet=112, Aktif mi?=True, 
Ajan 5: Servet=99, Aktif mi?=False, 
Ajan 6: Servet=139, Aktif mi?=True, 
Ajan 8: Servet=126, Aktif mi?=False, 
Ajan 9: Servet=104, Aktif mi?=False, 
Ajan 10: Servet=131, Aktif mi?=True, 
Ajan 11: Servet=137, Aktif mi?=True, 
Ajan 13: Servet=89, Aktif mi?=True, 
Ajan 15: Servet=99, Aktif mi?=False, 
Ajan 17: Servet=130, Aktif mi?=False, 
Ajan 20: Servet=92, Aktif mi?=True, 
Ajan 21: Servet=120, Aktif mi?=False, 
Ajan 22: Servet=103, Aktif mi?=False, 
Ajan 24: Servet=119, Aktif mi?=True, 
Ajan 25: Servet=136, Aktif mi?=True, 
Ajan 26: Servet=127, Aktif mi?=False, 
Ajan 27: Servet=135, Aktif mi?=True, 
Ajan 28: Servet=121, Aktif mi?=True, 
Ajan 29: Servet=95, Aktif mi?=False, 
Ajan 31: Servet=119, Aktif mi?=True, 
Ajan 32: Servet=111, Aktif mi?=True, 
Ajan 33: Servet=116, Aktif mi?=True, 
Ajan 34: Servet=95, Aktif mi?=True, 
Ajan 35: Servet=91, Aktif mi?=True, 
Ajan 36: Servet=133, Aktif mi?=False, 
Ajan 37: Serve